# QLoRA fine-tune — Qwen2.5-Coder-1.5B on BIRD (join-robust Text-to-SQL)

**Runtime → Change runtime type → T4 GPU** before running.

**Restart-safe:** the BIRD download and the training checkpoints are stored on Google Drive. If Colab disconnects, just reopen this notebook and **Run all** — data is reused and training **resumes from the last checkpoint** instead of starting over.

Edit `GH_USER`/`REPO` in the first cell if your GitHub repo name differs.

In [ ]:
import os
GH_USER = "goldeen802"
REPO = "bird-join-robust-text2sql"
if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/{GH_USER}/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!pip -q install -r requirements.txt
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime > Change runtime type > T4 GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = "/content/drive/MyDrive/bird-text2sql"
os.makedirs(f"{DRIVE}/bird_raw", exist_ok=True)
print("persisting data + checkpoints under", DRIVE)

In [ ]:
# Point the (git-ignored) data dir and the model output at Drive so nothing
# expensive is lost on a disconnect. Re-run safely each session.
import yaml, os
DRIVE = "/content/drive/MyDrive/bird-text2sql"
os.makedirs("data", exist_ok=True)
# symlink data/bird_raw -> Drive so BIRD is downloaded once and reused
!ln -sfn {DRIVE}/bird_raw data/bird_raw
# checkpoints + adapter -> Drive so training can resume after a disconnect
ql = yaml.safe_load(open("configs/qlora.yaml")); ql["output_dir"] = f"{DRIVE}/qwen-bird"
yaml.safe_dump(ql, open("configs/qlora.yaml", "w"), sort_keys=False)
pipe = yaml.safe_load(open("configs/pipeline.yaml")); pipe["model"]["adapter"] = f"{DRIVE}/qwen-bird"
yaml.safe_dump(pipe, open("configs/pipeline.yaml", "w"), sort_keys=False)
print("data/bird_raw -> Drive ; checkpoints -> Drive/qwen-bird")

In [ ]:
# Idempotent: download skips files already on Drive; subset/build are quick.
# If a URL 404s, update configs/bird_urls.yaml from https://bird-bench.github.io/
!python -m src.data.download
!python -m src.data.subset
!python -m src.data.build_training

In [ ]:
# Resumes automatically from the latest checkpoint on Drive if one exists.
!python -m src.train.train configs/qlora.yaml

In [ ]:
# Produces results/eval.json with the single- vs multi-table accuracy split.
!python scripts/run_eval.py --config configs/pipeline.yaml
import json
print(json.dumps(json.load(open("results/eval.json"))["summary"], indent=2))

In [ ]:
# The adapter already persists on Drive; this also gives you a direct download.
import shutil
shutil.make_archive("/content/qwen-bird", "zip", "/content/drive/MyDrive/bird-text2sql/qwen-bird")
from google.colab import files; files.download("/content/qwen-bird.zip")